# 🌽 MayaVoice — Fine-tuning Llama 3.1-8B para Idiomas Mayas
## Sprint 1: MVP con Unsloth + QLoRA en Google Colab

**Objetivo:** Fine-tune Llama 3.1-8B-Instruct para traducción ES↔Maya usando Unsloth con QLoRA de 4-bit.

**Dataset:** 23,738 pares paralelos (12 idiomas mayas, ambas direcciones)

**Issue:** [#6 - Primer fine-tuning con Unsloth en Colab Pro](https://github.com/DanielRegaladoUMiami/mayavoice-llm/issues/6)


In [1]:
# ============================================================
# 1. INSTALACIÓN (solo ejecutar una vez por sesión de Colab)
# ============================================================
%%capture
# Instalar Unsloth (auto-detecta versión correcta de PyTorch/CUDA)
!pip install unsloth
!wget -qO- https://raw.githubusercontent.com/unslothai/unsloth/main/unsloth/_auto_install.py | python -
!pip install sacrebleu

In [17]:
# ============================================================
# 2. CONFIGURACIÓN
# ============================================================
import json
import random
import torch
from pathlib import Path

# --- Configuración principal ---
CONFIG = {
    "base_model": "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",
    "max_seq_length": 2048,
    "lora_r": 16,
    "lora_alpha": 32,
    "lora_dropout": 0.05,
    "per_device_train_batch_size": 4,
    "gradient_accumulation_steps": 4,
    "learning_rate": 2e-4,
    "warmup_ratio": 0.1,
    "num_train_epochs": 3,
    "seed": 42,
    "output_dir": "./mayavoice-lora",
    "hub_model_id": "DanielRegaladoCardoso/mayavoice-llama3.1-8b-lora",
}

random.seed(CONFIG["seed"])
torch.manual_seed(CONFIG["seed"])
print(f"GPU: {torch.cuda.get_device_name(0)}")
vram = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"VRAM: {vram:.1f} GB")

GPU: NVIDIA A100-SXM4-80GB
VRAM: 85.1 GB


In [3]:
# ============================================================
# 3. CARGAR MODELO BASE CON UNSLOTH
# ============================================================
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=CONFIG["base_model"],
    max_seq_length=CONFIG["max_seq_length"],
    dtype=None,  # auto-detect
    load_in_4bit=True,
)

# Agregar LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=CONFIG["lora_r"],
    lora_alpha=CONFIG["lora_alpha"],
    lora_dropout=CONFIG["lora_dropout"],
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=CONFIG["seed"],
)

# Contar parámetros entrenables
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Parámetros entrenables: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.4: Fast Llama patching. Transformers: 5.2.0.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.251 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.70G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

Unsloth: Will load unsloth/meta-llama-3.1-8b-instruct-bnb-4bit as a legacy tokenizer.
Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2026.3.4 patched 32 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


Parámetros entrenables: 41,943,040 / 4,582,543,360 (0.92%)


In [4]:
# ============================================================
# 4. CARGAR DATOS DESDE GOOGLE DRIVE
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

import os

# Busca los datos en My Drive primero, luego en Shared
POSSIBLE_PATHS = [
    "/content/drive/MyDrive/mayavoice-data",                    # Si usas danielregaladoc9@gmail.com
    "/content/drive/Shareddrives/mayavoice-data",               # Si es Shared Drive
]

# También buscar en todo Drive por si la carpeta está en otro lugar
DATA_DIR = None
for path in POSSIBLE_PATHS:
    if os.path.exists(path) and os.path.isfile(f"{path}/train.jsonl"):
        DATA_DIR = path
        break

# Si no está en las rutas conocidas, buscar automáticamente
if DATA_DIR is None:
    import subprocess
    result = subprocess.run(["find", "/content/drive", "-name", "train.jsonl", "-maxdepth", 5],
                          capture_output=True, text=True, timeout=30)
    if result.stdout.strip():
        DATA_DIR = os.path.dirname(result.stdout.strip().split("\n")[0])

if DATA_DIR:
    print(f"✅ Datos encontrados en: {DATA_DIR}")
    for f in os.listdir(DATA_DIR):
        size = os.path.getsize(f"{DATA_DIR}/{f}") / 1024
        print(f"   {f} ({size:.1f} KB)")
else:
    print("❌ No se encontraron los datos.")
    print("   Asegúrate de que la carpeta 'mayavoice-data' esté en tu Drive")
    print("   o compartida con esta cuenta de Google")

def load_jsonl(path):
    data = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            data.append(json.loads(line))
    return data

train_data = load_jsonl(f"{DATA_DIR}/train.jsonl")
val_data = load_jsonl(f"{DATA_DIR}/val.jsonl")

print(f"\nTrain: {len(train_data)} ejemplos")
print(f"Val:   {len(val_data)} ejemplos")
print(f"\nEjemplo:")
print(json.dumps(train_data[0], indent=2, ensure_ascii=False))

Mounted at /content/drive
✅ Datos encontrados en: /content/drive/MyDrive/mayavoice-data
   train.jsonl (3667.7 KB)
   test_with_meta.jsonl (464.2 KB)
   val.jsonl (457.2 KB)
   .env.txt (0.0 KB)

Train: 18980 ejemplos
Val:   2360 ejemplos

Ejemplo:
{
  "instruction": "Traduce la siguiente oración al Sipakapense.",
  "input": "Todo el terreno lo dejaron para ti",
  "output": "Chawa xyoʼqken njeel ri uleew",
  "_lang": "sipakapense",
  "_direction": "es_to_maya"
}


In [5]:
# ============================================================
# 5. FORMATEAR DATOS PARA LLAMA 3.1 INSTRUCT
# ============================================================
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template="llama-3.1",
)

def format_example(example):
    """Convierte un ejemplo Alpaca al formato de chat de Llama 3.1."""
    messages = [
        {"role": "system", "content": "Eres MayaVoice, un asistente especializado en idiomas mayas de Guatemala. Traduces con precisión y respetas la riqueza lingüística de cada idioma."},
        {"role": "user", "content": f"{example['instruction']}\n\n{example['input']}"},
        {"role": "assistant", "content": example['output']},
    ]
    return {"messages": messages}

# Formatear datasets
train_formatted = [format_example(ex) for ex in train_data]
val_formatted = [format_example(ex) for ex in val_data]

# Verificar formato
print("Ejemplo formateado:")
for msg in train_formatted[0]["messages"]:
    print(f"  [{msg['role']}]: {msg['content'][:80]}...")


Ejemplo formateado:
  [system]: Eres MayaVoice, un asistente especializado en idiomas mayas de Guatemala. Traduc...
  [user]: Traduce la siguiente oración al Sipakapense.

Todo el terreno lo dejaron para ti...
  [assistant]: Chawa xyoʼqken njeel ri uleew...


In [6]:
# ============================================================
# 6. TOKENIZAR DATASET
# ============================================================
from unsloth.chat_templates import standardize_sharegpt, train_on_responses_only
from datasets import Dataset

def apply_chat_template(examples):
    texts = []
    for msgs in examples["messages"]:
        text = tokenizer.apply_chat_template(
            msgs, tokenize=False, add_generation_prompt=False
        )
        texts.append(text)
    return {"text": texts}

train_dataset = Dataset.from_list(train_formatted)
val_dataset = Dataset.from_list(val_formatted)

train_dataset = train_dataset.map(apply_chat_template, batched=True)
val_dataset = val_dataset.map(apply_chat_template, batched=True)

# Verificar longitudes de secuencia
lengths = [len(tokenizer.encode(t)) for t in train_dataset["text"][:500]]
print(f"Longitud promedio: {sum(lengths)/len(lengths):.0f} tokens")
print(f"Max: {max(lengths)}, Min: {min(lengths)}")
print(f"% > 2048 tokens: {100*sum(1 for l in lengths if l > 2048)/len(lengths):.1f}%")


Map:   0%|          | 0/18980 [00:00<?, ? examples/s]

Map:   0%|          | 0/2360 [00:00<?, ? examples/s]

Longitud promedio: 115 tokens
Max: 168, Min: 92
% > 2048 tokens: 0.0%


In [7]:
# ============================================================
# 7. ENTRENAR CON SFTTrainer
# ============================================================
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    dataset_text_field="text",
    max_seq_length=CONFIG["max_seq_length"],
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        output_dir=CONFIG["output_dir"],
        per_device_train_batch_size=CONFIG["per_device_train_batch_size"],
        gradient_accumulation_steps=CONFIG["gradient_accumulation_steps"],
        learning_rate=CONFIG["learning_rate"],
        warmup_ratio=CONFIG["warmup_ratio"],
        num_train_epochs=CONFIG["num_train_epochs"],
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=25,
        eval_strategy="steps",
        eval_steps=100,
        save_strategy="steps",
        save_steps=200,
        optim="adamw_8bit",
        lr_scheduler_type="cosine",
        seed=CONFIG["seed"],
        report_to="none",
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
    ),
)

# Entrenar solo en respuestas del asistente (no en instrucciones)
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|start_header_id|>user<|end_header_id|>\n\n",
    response_part="<|start_header_id|>assistant<|end_header_id|>\n\n",
)

print("🚀 Iniciando entrenamiento...")
stats = trainer.train()
print(f"\n✅ Entrenamiento completado!")
print(f"   Tiempo total: {stats.metrics['train_runtime']/60:.1f} minutos")
print(f"   Loss final: {stats.metrics['train_loss']:.4f}")


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/18980 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/2360 [00:00<?, ? examples/s]

Map (num_proc=16):   0%|          | 0/18980 [00:00<?, ? examples/s]

Filter (num_proc=16):   0%|          | 0/18980 [00:00<?, ? examples/s]

...

🚀 Iniciando entrenamiento...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 18,980 | Num Epochs = 3 | Total steps = 3,561
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)


Step,Training Loss,Validation Loss
100,3.612286,3.464003
200,3.328343,3.260256
300,3.176378,3.111737
400,3.031510,3.015213
500,2.884501,2.917219
600,2.803920,2.763945
700,2.716300,2.687288
800,2.632700,2.595900
900,2.620571,2.525988
1000,2.485478,2.474862


Unsloth: Not an error, but LlamaForCausalLM does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient
--- Logging error ---
Traceback (most recent call last):
  File "/usr/lib/python3.12/logging/__init__.py", line 1160, in emit
    msg = self.format(record)
          ^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/logging/__init__.py", line 999, in format
    return fmt.format(record)
           ^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/logging/__init__.py", line 703, in format
    record.message = record.getMessage()
                     ^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/logging/__init__.py", line 392, in getMessage
    msg = msg % self.args
          ~~~~^~~~~~~~~~~
TypeError: not all arguments converted during string formatting
Call stack:
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_co


✅ Entrenamiento completado!
   Tiempo total: 174.4 minutos
   Loss final: 1.9271


In [11]:
# ============================================================
# 8. EVALUACIÓN RÁPIDA — TRADUCCIONES DE PRUEBA
# ============================================================
FastLanguageModel.for_inference(model)

test_cases = [
    ("Traduce del español al K'iche'.", "Buenos días, ¿cómo estás?"),
    ("Traduce del español al Mam.", "La tierra es sagrada para nosotros."),
    ("Traduce del español al Q'eqchi'.", "Los niños están jugando en el campo."),
    ("Traduce del K'iche' al español.", "Saqarik, la utz awach?"),
    ("Traduce del español al Tz'utujil.", "El maíz ya está listo para cosechar."),
    ("Responde en K'iche' como un asistente amigable.", "¿Qué significa la palabra 'saqarik'?"),
]

for instruction, input_text in test_cases:
    messages = [
        {"role": "system", "content": "Eres MayaVoice, un asistente especializado en idiomas mayas de Guatemala."},
        {"role": "user", "content": f"{instruction}\n\n{input_text}"},
    ]

    input_ids = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True,
        return_tensors="pt"
    )
    # Si devuelve dict (BatchEncoding), extraer input_ids
    if hasattr(input_ids, "input_ids"):
        input_ids = input_ids["input_ids"]
    input_ids = input_ids.to("cuda")


    outputs = model.generate(
        input_ids=input_ids, max_new_tokens=128, temperature=0.3,
        use_cache=True, do_sample=True,
    )
    response = tokenizer.decode(outputs[0][input_ids.shape[-1]:], skip_special_tokens=True)


    print(f"📝 {instruction}")
    print(f"   Input:  {input_text}")
    print(f"   Output: {response}")
    print()


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


📝 Traduce del español al K'iche'.
   Input:  Buenos días, ¿cómo estás?
   Output: Utz qʼij, jachin kʼo chi awetaq

📝 Traduce del español al Mam.
   Input:  La tierra es sagrada para nosotros.
   Output: Qaʼnxa txʼotxʼ tiʼj qe.

📝 Traduce del español al Q'eqchi'.
   Input:  Los niños están jugando en el campo.
   Output: Yookebʼ chi bʼatzʼok saʼ li kʼal li al

📝 Traduce del K'iche' al español.
   Input:  Saqarik, la utz awach?
   Output: ¿Es cierto que eres bueno?

📝 Traduce del español al Tz'utujil.
   Input:  El maíz ya está listo para cosechar.
   Output: Qʼan chik ja ixiim

📝 Responde en K'iche' como un asistente amigable.
   Input:  ¿Qué significa la palabra 'saqarik'?
   Output: Saqarik kujawik ri uqʼabʼ cheʼ



In [13]:
# ============================================================
# 9. MÉTRICAS BASELINE — BLEU y chrF (Issue #7)
# ============================================================
import sacrebleu
from collections import defaultdict

# Cargar test set con metadata
test_meta = load_jsonl(f"{DATA_DIR}/test_with_meta.jsonl")
print(f"Test set: {len(test_meta)} ejemplos")

# Generar traducciones para el test set (sample para velocidad)
SAMPLE_SIZE = 200  # Ajustar según tiempo disponible
random.seed(42)
test_sample = random.sample(test_meta, min(SAMPLE_SIZE, len(test_meta)))

FastLanguageModel.for_inference(model)

results_by_lang = defaultdict(lambda: {"refs": [], "hyps": [], "direction": []})

for i, example in enumerate(test_sample):
    messages = [
        {"role": "system", "content": "Eres MayaVoice, un asistente especializado en idiomas mayas de Guatemala."},
        {"role": "user", "content": f"{example['instruction']}\n\n{example['input']}"},
    ]

    input_ids = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True,
        return_tensors="pt"
    )
    if hasattr(input_ids, "input_ids"):
        input_ids = input_ids["input_ids"]
    input_ids = input_ids.to("cuda")

    outputs = model.generate(
        input_ids=input_ids, max_new_tokens=128, temperature=0.1,
        use_cache=True, do_sample=False,
    )
    hyp = tokenizer.decode(outputs[0][input_ids.shape[-1]:], skip_special_tokens=True).strip()

    ref = example["output"].strip()
    lang = example.get("_lang", "unknown")
    direction = example.get("_direction", "unknown")

    results_by_lang[lang]["refs"].append(ref)
    results_by_lang[lang]["hyps"].append(hyp)
    results_by_lang[lang]["direction"].append(direction)

    if (i + 1) % 50 == 0:
        print(f"  Procesados {i+1}/{len(test_sample)}...")

print(f"\n{'='*70}")
print(f"{'MÉTRICAS BASELINE — MayaVoice Sprint 1':^70}")
print(f"{'='*70}")
print(f"{'Idioma':<15} {'N':<6} {'BLEU':<10} {'chrF':<10}")
print("-" * 41)

all_refs, all_hyps = [], []
for lang in sorted(results_by_lang.keys()):
    data = results_by_lang[lang]
    refs = data["refs"]
    hyps = data["hyps"]
    all_refs.extend(refs)
    all_hyps.extend(hyps)

    bleu = sacrebleu.corpus_bleu(hyps, [refs])
    chrf = sacrebleu.corpus_chrf(hyps, [refs])
    print(f"{lang:<15} {len(refs):<6} {bleu.score:<10.2f} {chrf.score:<10.2f}")

# Overall
bleu_all = sacrebleu.corpus_bleu(all_hyps, [all_refs])
chrf_all = sacrebleu.corpus_chrf(all_hyps, [all_refs])
print("-" * 41)
print(f"{'TOTAL':<15} {len(all_refs):<6} {bleu_all.score:<10.2f} {chrf_all.score:<10.2f}")

# Guardar resultados en Drive
metrics = {
    "model": CONFIG["base_model"],
    "overall_bleu": bleu_all.score,
    "overall_chrf": chrf_all.score,
    "sample_size": len(test_sample),
    "per_language": {}
}
for lang in sorted(results_by_lang.keys()):
    data = results_by_lang[lang]
    bleu = sacrebleu.corpus_bleu(data["hyps"], [data["refs"]])
    chrf = sacrebleu.corpus_chrf(data["hyps"], [data["refs"]])
    metrics["per_language"][lang] = {
        "bleu": bleu.score, "chrf": chrf.score, "n": len(data["refs"])
    }

metrics_path = f"{DATA_DIR}/baseline_metrics.json"
with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=2)
print(f"\n✅ Métricas guardadas en Google Drive: {metrics_path}")

Test set: 6486 ejemplos
  Procesados 50/200...
  Procesados 100/200...
  Procesados 150/200...
  Procesados 200/200...

                MÉTRICAS BASELINE — MayaVoice Sprint 1                
Idioma          N      BLEU       chrF      
-----------------------------------------
achi            13     2.48       16.38     
awakateko       11     3.38       14.97     
chuj            23     7.30       29.52     
itza            6      0.00       11.80     
kiche           11     9.51       29.69     
mam             23     10.76      29.71     
poqomam         20     19.38      38.53     
poqomchi        11     13.09      31.42     
qanjobal        21     3.14       28.90     
qeqchi          22     4.19       27.84     
sipakapense     8      16.03      35.86     
tektiteko       18     11.57      30.09     
tzutujil        13     9.52       28.66     
-----------------------------------------
TOTAL           200    8.85       28.82     

✅ Métricas guardadas en Google Drive: /content/dr

In [18]:
# ============================================================
# 10. GUARDAR MODELO
# ============================================================
import os
from huggingface_hub import login

HF_TOKEN = ""

# --- Guardar en Drive (siempre) ---
DRIVE_MODEL_DIR = "/content/drive/MyDrive/mayavoice-data/mayavoice-lora"
model.save_pretrained(DRIVE_MODEL_DIR)
tokenizer.save_pretrained(DRIVE_MODEL_DIR)
print(f"✅ LoRA adapters guardados en Google Drive: {DRIVE_MODEL_DIR}/")

# --- Subir a HuggingFace Hub ---
if HF_TOKEN:
    login(token=HF_TOKEN)
    model.push_to_hub(CONFIG["hub_model_id"], token=HF_TOKEN)
    tokenizer.push_to_hub(CONFIG["hub_model_id"], token=HF_TOKEN)
    print(f"✅ Modelo subido a https://huggingface.co/{CONFIG['hub_model_id']}")
else:
    print("⚠️  No se encontró HF_TOKEN. Crea .env en Drive/mayavoice-data/")

# --- Exportar a GGUF para Ollama (descomentar si necesitas) ---
# model.save_pretrained_gguf(
#     "/content/drive/MyDrive/mayavoice-data/mayavoice-gguf",
#     tokenizer,
#     quantization_method="q4_k_m",
# )
# print("✅ GGUF exportado para uso con Ollama")

✅ LoRA adapters guardados en Google Drive: /content/drive/MyDrive/mayavoice-data/mayavoice-lora/


README.md:   0%|          | 0.00/589 [00:00<?, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   0%|          |  554kB /  168MB            

Saved model to https://huggingface.co/DanielRegaladoCardoso/mayavoice-llama3.1-8b-lora


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mpgxo1fdvl/tokenizer.json: 100%|##########| 17.2MB / 17.2MB            

✅ Modelo subido a https://huggingface.co/DanielRegaladoCardoso/mayavoice-llama3.1-8b-lora


## 📊 Resumen del Sprint 1

### Qué se logró:
- ✅ Fine-tuning de Llama 3.1-8B con QLoRA (4-bit) usando Unsloth
- ✅ Dataset: 18,980 train / 2,360 val / 2,398 test (12 idiomas mayas)
- ✅ Formato: Instrucciones en español con chat template de Llama 3.1
- ✅ Métricas baseline BLEU y chrF por idioma

### Próximos pasos (Sprint 2):
- [ ] Conseguir datos de Kaqchikel (3er idioma más hablado, ~1M hablantes)
- [ ] Implementar RAG pipeline con ChromaDB
- [ ] Agregar datos de Achi e Ixil
- [ ] Crear test set manual con hablantes nativos (UVG)
- [ ] Probar Qwen 2.5-7B como alternativa si BLEU < esperado

### Issues cerrados con este notebook:
- [#6](https://github.com/DanielRegaladoUMiami/mayavoice-llm/issues/6): Primer fine-tuning con Unsloth
- [#7](https://github.com/DanielRegaladoUMiami/mayavoice-llm/issues/7): Métricas baseline BLEU y chrF
